# Clima de las regiones de la Tierra y de los sitios de Marte (v0)
**NASA Space Apps Challenge 2026** · Capa de clima para comparar cada sitio de Marte con las regiones candidatas y los análogos
conocidos de la Tierra.

| Lado | Datos | Qué se mide |
|---|---|---|
| Tierra | MERRA-2 2014–2023 (ya en Drive, de `EDA_analogos_v1.3`) + MERRA-2 `M2SMNXSLV` (máximas y mínimas diarias) | temperatura media, del mes más frío y del más cálido, mínima de las noches del mes más frío, oscilación diaria, precipitación |
| Marte | Mars Climate Database (LMD/ESA/CNES), interfaz web | temperatura del aire a 2 m a lo largo del día en 4 estaciones (Ls 0°, 90°, 180°, 270°) y presión |
| Luna | — | no tiene atmósfera: su temperatura depende de la iluminación, no del clima. Solo se guarda el clima de las regiones de la Tierra del perfil Luna, como dato informativo |

**Comparación por percentiles relativos**: ningún lugar de la Tierra es tan frío como Marte (media de unos −60 °C), así que no se
comparan valores absolutos. Cada sitio de Marte se coloca en su percentil dentro de los sitios de Marte (p. ej. «entre el 10 % más frío»)
y cada región de la Tierra en su percentil dentro de las regiones de la Tierra; se emparejan los que ocupan el mismo puesto relativo.
Distancia de clima = media de las diferencias de percentil (0 a 1) en 4 medidas: temperatura media, mínima del mes más frío,
amplitud anual y oscilación diaria. Se combina con la distancia de topografía de `topografia_sitios_v1`.

**Por tandas**: la descarga de MERRA-2 (120 archivos pequeños) y las consultas a la MCD (59 sitios × 4 estaciones) se guardan una a una
en Drive y cada ejecución se para a los `MAX_MINUTOS`. **Si al final pone «⏸ Quedan…», vuelve a ejecutar el notebook**: sigue donde lo dejó.

## 0. Instalación

In [1]:
!pip install -q earthaccess

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.


## 1. Configuración

In [2]:
import os, re, gc, json, time, glob, datetime, unicodedata
from urllib.parse import urljoin
import numpy as np, pandas as pd, requests, xarray as xr
import matplotlib.pyplot as plt
import earthaccess
from shapely import contains_xy
from shapely.geometry import shape, Point
from shapely import affinity
from IPython.display import display
from google.colab import drive, userdata

drive.mount("/content/drive")
for raiz in ["/content/drive/MyDrive", "/content/drive/My Drive"]:
    if os.path.isdir(raiz):
        break
BASE = f"{raiz}/Hackathon Nasa"
VERSION = "clima_v0"
RES, WEB = f"{BASE}/04_resultados/{VERSION}", f"{BASE}/06_web/{VERSION}"
CACHE_M2 = f"{BASE}/03_datos/cache_api/clima/merra2_diaria"
CACHE_MCD = f"{BASE}/03_datos/cache_api/clima/mcd"
TMP = "/content/tmp/clima"
for d in (RES, WEB, CACHE_M2, CACHE_MCD, TMP):
    os.makedirs(d, exist_ok=True)

MAX_MINUTOS = 40                       # tiempo máximo por ejecución (vuelve a ejecutar para seguir)
ANIOS = range(2014, 2024)              # mismo periodo que el índice de Marte v1.3
GRADIENTE_K_KM = 6.5                   # corrección de temperatura por altitud (celda de 50 km → altura real del lugar)
RADIO_ANALOGO_KM = 10
LS_MARTE = [0, 90, 180, 270]           # primavera, verano, otoño, invierno del hemisferio norte de Marte
ALTURA_MARTE_M = 2                     # temperatura del aire a 2 m, como T2M de MERRA-2
URLS_MCD = ["https://www-mars.lmd.jussieu.fr/mcd_python/cgi-bin/mcdcgi.py",
            "http://www-mars.lmd.jussieu.fr/mcd_python/cgi-bin/mcdcgi.py"]
MEDIDAS = ["t_media", "t_min", "amplitud_anual", "osc_diaria"]   # las que entran en la comparación
UMBRAL_PARECIDO = 0.2
T0 = time.time()
minutos = lambda: (time.time() - T0) / 60

FIGURAS = f"{RES}/figuras"; os.makedirs(FIGURAS, exist_ok=True)
_mostrar = getattr(plt, "_show_original", plt.show); plt._show_original = _mostrar
_n_fig = [0]
def _mostrar_y_guardar(*a, **k):
    fig = plt.gcf()
    if fig.axes:
        _n_fig[0] += 1
        titulo = next((ax.get_title() for ax in fig.axes if ax.get_title()), "figura")
        slug = re.sub(r"[^a-zA-Z0-9]+", "_", unicodedata.normalize("NFKD", titulo).encode("ascii", "ignore").decode()).strip("_").lower()[:50]
        fig.savefig(f"{FIGURAS}/fig_{_n_fig[0]:02d}_{slug}.png", dpi=80, bbox_inches="tight")
    _mostrar(*a, **k)
plt.show = _mostrar_y_guardar

os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")
auth = earthaccess.login(strategy="environment")
SESION = earthaccess.get_requests_https_session()
entrada = json.load(open(f"{BASE}/06_web/manifest.json", encoding="utf-8"))
print("Earthdata:", auth.authenticated)

Mounted at /content/drive
Earthdata: True


## 2. Tierra: regiones candidatas y análogos conocidos
Las mismas que en `topografia_v1` (mismos `region_id`). Cada una se asigna a las celdas de MERRA-2 (0,5° × 0,625°) cuyo centro cae
dentro; los análogos (círculo de 10 km) y las regiones muy pequeñas usan la celda más cercana.

In [3]:
def circulo_grados(lon, lat, radio_km):
    rlat = radio_km / 111.32
    return affinity.scale(Point(lon, lat).buffer(rlat, 64), xfact=1 / max(np.cos(np.radians(lat)), 0.05), yfact=1)

filas = []
for cuerpo in ["luna", "marte"]:
    info = entrada["cuerpos"][cuerpo]
    for ft in json.load(open(f"{BASE}/06_web/{info['carpeta']}/regiones.geojson", encoding="utf-8"))["features"]:
        p = ft["properties"]
        filas.append({"region_id": p["id"], "cuerpo": cuerpo, "tipo": "region_candidata", "region": p.get("region"), "geom": shape(ft["geometry"])})
    for ft in json.load(open(f"{BASE}/06_web/{info['carpeta']}/analogos.geojson", encoding="utf-8"))["features"]:
        p = ft["properties"]
        ok = (p.get("tipo") in ("analogo_luna", "impacto")) if cuerpo == "luna" else (p.get("verificacion") == "verificada")
        if not ok or not ft.get("geometry"):
            continue
        lon, lat = ft["geometry"]["coordinates"][:2]
        slug = re.sub(r"[^a-z0-9]+", "-", unicodedata.normalize("NFKD", p["nombre"]).encode("ascii", "ignore").decode().lower()).strip("-")[:40]
        filas.append({"region_id": f"analogo-{cuerpo}-{slug}", "cuerpo": cuerpo, "tipo": "analogo_conocido", "region": p["nombre"],
                      "geom": circulo_grados(lon, lat, RADIO_ANALOGO_KM)})
ent = pd.DataFrame(filas)

# Altura real de cada lugar (media de ASTER a 30 m, de topografia_v1) para corregir la temperatura de la celda
ruta_t1 = f"{BASE}/06_web/topografia_v1/topografia.json"
topo1 = json.load(open(ruta_t1, encoding="utf-8"))["regiones"] if os.path.exists(ruta_t1) else {}
ent["elev_real_m"] = ent.region_id.map(lambda r: (topo1.get(r) or {}).get("elev_media_m"))

clima_v13 = xr.open_dataset(f"{BASE}/03_datos/procesados/clima_merra2_2014_2023.nc").load()
print("Variables del clima de v1.3:", list(clima_v13.data_vars))
LAT, LON = clima_v13.lat.values, clima_v13.lon.values
LL, LA = np.meshgrid(LON, LAT)
celdas = {}
for r in ent.itertuples():
    dentro = contains_xy(r.geom, LL, LA)
    if dentro.any():
        celdas[r.region_id] = np.argwhere(dentro)
    else:
        c = r.geom.centroid
        celdas[r.region_id] = np.array([[np.abs(LAT - c.y).argmin(), np.abs(((LON - c.x + 180) % 360) - 180).argmin()]])
ent["n_celdas"] = ent.region_id.map(lambda r: len(celdas[r]))
print(f"{len(ent)} lugares de la Tierra:", ent.groupby(["cuerpo", "tipo"]).size().to_dict())

Variables del clima de v1.3: ['T_media', 'T_frio', 'T_calido', 'P_anual', 'tierra', 'hielo', 'elevacion', 'presion_aprox', 'ndvi_max', 'frac_desnudo']
56 lugares de la Tierra: {('luna', 'analogo_conocido'): 7, ('luna', 'region_candidata'): 20, ('marte', 'analogo_conocido'): 8, ('marte', 'region_candidata'): 21}


## 3. Tierra: temperatura media, del mes más frío y del más cálido, y precipitación (MERRA-2, ya calculado en v1.3)
Se corrige por altitud: la celda de MERRA-2 tiene la altura media de 50 km, que puede diferir mucho del lugar
(Mauna Kea, Tso Kar). Corrección: −6,5 °C por cada km que el lugar está por encima de la celda.

In [4]:
def en_celdas(da, rid):
    ij = celdas[rid]
    return float(np.nanmean(da.values[ij[:, 0], ij[:, 1]]))

a_celsius = (lambda v: v - 273.15) if float(clima_v13["T_media"].max()) > 150 else (lambda v: v)
tierra = ent.drop(columns="geom").copy()
for var, col in (("T_media", "t_media"), ("T_frio", "t_frio"), ("T_calido", "t_calido")):
    tierra[col] = [a_celsius(en_celdas(clima_v13[var], r)) for r in tierra.region_id]
tierra["precip_mm"] = [en_celdas(clima_v13["P_anual"], r) for r in tierra.region_id] if "P_anual" in clima_v13 else np.nan
tierra["elev_celda_m"] = [en_celdas(clima_v13["elevacion"], r) for r in tierra.region_id]
tierra["correccion_c"] = (-GRADIENTE_K_KM * (tierra.elev_real_m.astype(float) - tierra.elev_celda_m) / 1000).fillna(0).round(1)
for col in ("t_media", "t_frio", "t_calido"):
    tierra[col] = (tierra[col] + tierra.correccion_c).round(1)
tierra["amplitud_anual"] = (tierra.t_calido - tierra.t_frio).round(1)
display(tierra[["region_id", "region", "elev_celda_m", "elev_real_m", "correccion_c", "t_media", "t_frio", "t_calido", "precip_mm"]].round(1))

,region_id,region,elev_celda_m,elev_real_m,correccion_c,t_media,t_frio,t_calido,precip_mm
0,luna-1,"Afar, Etiopía",362.0,290,0.5,30.8,26.1,35.1,334.5
1,luna-2,"Gobernacion de Rif Dimashq, Siria",786.6,803,-0.1,18.8,7.0,29.3,102.7
2,luna-3,"Provincia de Medina, Arabia Saudita",1304.9,1105,1.3,24.6,13.4,33.8,38.9
3,luna-4,"Afar, Etiopía",477.5,477,0.0,29.7,25.0,34.6,441.8
4,luna-5,"Condado de Marsabit, Kenia",569.4,482,0.6,29.8,28.5,31.3,214.9
5,luna-6,"Provincia de La Meca, Arabia Saudita",1291.6,1254,0.2,25.7,17.6,32.5,74.4
6,luna-7,Islandia,801.8,648,1.0,-1.9,-6.6,3.7,875.2
7,luna-8,"Afar, Etiopía",564.8,466,0.6,29.7,25.2,34.1,545.8
8,luna-9,"Provincia de Tabuk, Arabia Saudita",1295.0,1424,-0.8,19.8,8.6,29.2,29.4
9,luna-10,"Tibesti, Chad",1666.8,1681,-0.1,20.2,11.7,26.5,51.5


## 4. Tierra: oscilación diaria (MERRA-2 `M2SMNXSLV`, medias mensuales de las máximas y mínimas diarias, 2014–2023)
120 archivos mensuales: se descarga uno, se extraen los valores de los lugares, se borra y se guarda un resumen pequeño en Drive.

In [5]:
meses = [f"{a}{m:02d}" for a in ANIOS for m in range(1, 13)]
hecho = lambda ym: os.path.exists(f"{CACHE_M2}/{ym}.npz")
pendientes = [ym for ym in meses if not hecho(ym)]
print(f"Meses ya procesados: {len(meses) - len(pendientes)}/{len(meses)}")
if pendientes:
    granulos = earthaccess.search_data(short_name="M2SMNXSLV", version="5.12.4",
                                       temporal=(f"{min(ANIOS)}-01-01", f"{max(ANIOS)}-12-31"))
    enlaces = {}
    for g in granulos:
        for u in g.data_links():
            m = re.search(r"\.(\d{6})\.nc4$", u)
            if m:
                enlaces[m.group(1)] = u
    print(f"Granulos encontrados: {len(enlaces)}")
    for k, ym in enumerate(pendientes):
        if minutos() > MAX_MINUTOS:
            break
        if ym not in enlaces:
            print("  ⚠ falta el mes", ym); continue
        ruta = f"{TMP}/{ym}.nc4"
        try:
            with SESION.get(enlaces[ym], stream=True, timeout=300) as r:
                r.raise_for_status()
                with open(ruta, "wb") as f:
                    for t in r.iter_content(1 << 22):
                        f.write(t)
            with xr.open_dataset(ruta) as ds:
                tmax = ds["T2MMAX"].isel(time=0).values; tmin = ds["T2MMIN"].isel(time=0).values
                assert tmax.shape == LL.shape, f"Rejilla distinta de la de v1.3: {tmax.shape}"
            vals = {rid: (np.nanmean(tmax[ij[:, 0], ij[:, 1]]), np.nanmean(tmin[ij[:, 0], ij[:, 1]])) for rid, ij in celdas.items()}
            np.savez(f"{CACHE_M2}/{ym}.npz", ids=np.array(list(vals)), tmax=np.array([v[0] for v in vals.values()]),
                     tmin=np.array([v[1] for v in vals.values()]))
            del tmax, tmin
        except Exception as e:
            print(f"  ⚠ {ym}: {e!r} (se reintentará)")
        finally:
            if os.path.exists(ruta): os.remove(ruta)
            gc.collect()
        if k % 20 == 0:
            print(f"  {ym} · {minutos():.0f} min")
FALTAN_M2 = [ym for ym in meses if not hecho(ym)]

# Climatología mensual (media de los 10 años por mes) de máximas y mínimas diarias
acum = {}
for ym in meses:
    if not hecho(ym):
        continue
    d = np.load(f"{CACHE_M2}/{ym}.npz")
    for rid, mx, mn in zip(d["ids"], d["tmax"], d["tmin"]):
        acum.setdefault(str(rid), {}).setdefault(int(ym[4:]), []).append((mx, mn))
def resumen_diario(rid):
    if rid not in acum:
        return np.nan, np.nan
    clim = {m: np.mean(v, axis=0) for m, v in acum[rid].items()}
    osc = np.mean([mx - mn for mx, mn in clim.values()])
    t_min = min(mn for mx, mn in clim.values()) - 273.15          # mínima diaria media del mes más frío
    return osc, t_min
res = [resumen_diario(r) for r in tierra.region_id]
tierra["osc_diaria"] = [round(o, 1) for o, _ in res]
tierra["t_min"] = [round(t + c, 1) for (_, t), c in zip(res, tierra.correccion_c)]
print("✅ MERRA-2 diario completo." if not FALTAN_M2 else f"⏸ Quedan {len(FALTAN_M2)} meses de MERRA-2: vuelve a ejecutar el notebook.")
tierra.to_csv(f"{RES}/clima_tierra.csv", index=False)
display(tierra[["region", "cuerpo", "tipo", "t_media", "t_min", "amplitud_anual", "osc_diaria", "precip_mm"]])

Meses ya procesados: 0/120
Granulos encontrados: 120
  ⚠ 201401: HTTPError('502 Server Error: Bad Gateway for url: https://data.gesdisc.earthdata.nasa.gov/data/MERRA2_MONTHLY/M2SMNXSLV.5.12.4/2014/MERRA2_400.statM_2d_slv_Nx.201401.nc4') (se reintentará)
  201401 · 0 min
  201509 · 1 min
  201705 · 2 min
  201901 · 2 min
  202009 · 3 min
  202205 · 3 min
⏸ Quedan 1 meses de MERRA-2: vuelve a ejecutar el notebook.


,region,cuerpo,tipo,t_media,t_min,amplitud_anual,osc_diaria,precip_mm
0,"Afar, Etiopía",luna,region_candidata,30.8,19.700001,9.0,13.600000,334.461535
1,"Gobernacion de Rif Dimashq, Siria",luna,region_candidata,18.8,1.500000,22.3,15.800000,102.652701
2,"Provincia de Medina, Arabia Saudita",luna,region_candidata,24.6,6.100000,20.4,16.100000,38.906014
3,"Afar, Etiopía",luna,region_candidata,29.7,17.900000,9.6,14.100000,441.847235
4,"Condado de Marsabit, Kenia",luna,region_candidata,29.8,21.900000,2.8,13.100000,214.893535
5,"Provincia de La Meca, Arabia Saudita",luna,region_candidata,25.7,9.900000,14.9,16.400000,74.417481
6,Islandia,luna,region_candidata,-1.9,-9.700000,10.3,4.200000,875.228624
7,"Afar, Etiopía",luna,region_candidata,29.7,17.900000,8.9,14.400000,545.757902
8,"Provincia de Tabuk, Arabia Saudita",luna,region_candidata,19.8,1.800000,20.6,16.100000,29.377928
9,"Tibesti, Chad",luna,region_candidata,20.2,3.400000,14.8,16.900000,51.475073


## 5. Marte: Mars Climate Database (MCD)
Para cada sitio y cada estación se pide a la interfaz web de la MCD el ciclo diario (25 horas locales) de la temperatura del aire a
2 m y la presión en superficie, con el escenario de polvo climatológico y la topografía de alta resolución. Cada respuesta se guarda en
`03_datos/cache_api/clima/mcd/`. Primero se prueba el servicio con un sitio: si no responde, se salta Marte y el resto del notebook sigue.

In [6]:
sitios = json.load(open(f"{BASE}/06_web/sitios_v0/sitios.geojson", encoding="utf-8"))["features"]
marte = [{"id": f["properties"]["id"], "nombre": f["properties"]["nombre"], "tipo": f["properties"]["tipo"],
          "lon": ((f["geometry"]["coordinates"][0] + 180) % 360) - 180, "lat": f["geometry"]["coordinates"][1]}
         for f in sitios if f["properties"]["cuerpo"] == "marte" and f["geometry"]["type"] == "Point"]

def parametros(lat, lon, ls):
    return {"var1": "t", "var2": "ps", "var3": "none", "var4": "none", "datekeyhtml": "1", "ls": f"{ls:.1f}",
            "localtime": "all", "latitude": f"{lat:.3f}", "longitude": f"{lon:.3f}", "altitude": str(ALTURA_MARTE_M), "zkey": "3",
            "isfixedlt": "off", "dust": "1", "hrkey": "1", "averaging": "off", "dpi": "80", "islog": "off", "colorm": "jet",
            "minval": "", "maxval": "", "proj": "cyl", "plat": "", "plon": "", "trans": "", "iswind": "off",
            "latpoint": "", "lonpoint": "", "spacecraft": "none"}

def leer_ascii(txt):
    """Bloques de la salida ASCII de la MCD: [(etiqueta, x, y), ...] en el orden de var1, var2…"""
    bloques, cur = [], None
    for linea in txt.splitlines():
        if linea.startswith("#"):
            if cur and cur["x"]:
                bloques.append(cur); cur = None
            if cur is None:
                cur = {"lab": "", "x": [], "y": []}
            if "Column 2" in linea:
                cur["lab"] = linea.split("is", 1)[-1].strip()
            continue
        partes = linea.split()
        if len(partes) == 2 and cur is not None:
            try:
                x, y = float(partes[0]), float(partes[1])
            except ValueError:
                continue
            cur["x"].append(x); cur["y"].append(y)
    if cur and cur["x"]:
        bloques.append(cur)
    return bloques

ULTIMO_HTML = [""]
def consulta_mcd(lat, lon, ls):
    errores = []
    for url in URLS_MCD:
        try:
            r = requests.get(url, params=parametros(lat, lon, ls), timeout=180)
            ULTIMO_HTML[0] = r.text
            m = re.search(r"""href=['"]?([^'" >]*txt/[^'" >]+\.txt)""", r.text)
            if r.status_code != 200 or not m:
                errores.append(f"{url}: HTTP {r.status_code}, sin enlace al .txt"); continue
            t = requests.get(urljoin(url, m.group(1)), timeout=120)
            bl = leer_ascii(t.text)
            if len(bl) >= 1 and len(bl[0]["x"]) >= 10:
                return t.text
            errores.append(f"{url}: .txt sin datos")
        except requests.RequestException as e:
            errores.append(f"{url}: {e!r}")
    raise RuntimeError(" | ".join(errores))

def ruta_mcd(sid, ls):
    return f"{CACHE_MCD}/{sid}__ls{ls:03d}.txt"

MCD_OK = True
s0 = marte[0]
if not os.path.exists(ruta_mcd(s0["id"], 90)):
    try:
        txt = consulta_mcd(s0["lat"], s0["lon"], 90)
        open(ruta_mcd(s0["id"], 90), "w").write(txt)
    except Exception as e:
        MCD_OK = False
        print("❌ La MCD no responde como se esperaba:", e)
        print("Inicio de la última respuesta (cópialo en el chat si falla):\n", ULTIMO_HTML[0][:1500])
if MCD_OK:
    bl = leer_ascii(open(ruta_mcd(s0["id"], 90)).read())
    print(f"Prueba con {s0['nombre']} (Ls 90°):", [(b["lab"][:40], len(b["x"]), round(min(b["y"]), 1), round(max(b["y"]), 1)) for b in bl])

❌ La MCD no responde como se esperaba: https://www-mars.lmd.jussieu.fr/mcd_python/cgi-bin/mcdcgi.py: ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host='www-mars.lmd.jussieu.fr', port=443): Max retries exceeded with url: /mcd_python/cgi-bin/mcdcgi.py?var1=t&var2=ps&var3=none&var4=none&datekeyhtml=1&ls=90.0&localtime=all&latitude=-45.000&longitude=-158.000&altitude=2&zkey=3&isfixedlt=off&dust=1&hrkey=1&averaging=off&dpi=80&islog=off&colorm=jet&minval=&maxval=&proj=cyl&plat=&plon=&trans=&iswind=off&latpoint=&lonpoint=&spacecraft=none (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7d3f4972fd90>, 'Connection to www-mars.lmd.jussieu.fr timed out. (connect timeout=180)'))")) | http://www-mars.lmd.jussieu.fr/mcd_python/cgi-bin/mcdcgi.py: ConnectTimeout(MaxRetryError("HTTPConnectionPool(host='www-mars.lmd.jussieu.fr', port=80): Max retries exceeded with url: /mcd_python/cgi-bin/mcdcgi.py?var1=t&var2=ps&var3=none&var4=none&datekeyhtml=1&ls=90.0&localtime=all&

In [7]:
if MCD_OK:
    trabajo = [(s, ls) for s in marte for ls in LS_MARTE if not os.path.exists(ruta_mcd(s["id"], ls))]
    print(f"Consultas pendientes: {len(trabajo)} de {len(marte) * len(LS_MARTE)}")
    fallos = 0
    for k, (s, ls) in enumerate(trabajo):
        if minutos() > MAX_MINUTOS:
            break
        try:
            open(ruta_mcd(s["id"], ls), "w").write(consulta_mcd(s["lat"], s["lon"], ls))
        except Exception as e:
            fallos += 1
            print(f"  ⚠ {s['nombre']} Ls {ls}: {str(e)[:150]} (se reintentará)")
            if fallos >= 5 and fallos == k + 1:
                print("  Demasiados fallos seguidos: se para Marte en esta ejecución."); break
        time.sleep(1)                                                      # no saturar el servidor
        if k % 25 == 0:
            print(f"  {k + 1}/{len(trabajo)} · {minutos():.0f} min")
    FALTAN_MCD = [(s["id"], ls) for s in marte for ls in LS_MARTE if not os.path.exists(ruta_mcd(s["id"], ls))]
    print("✅ MCD completa." if not FALTAN_MCD else f"⏸ Quedan {len(FALTAN_MCD)} consultas a la MCD: vuelve a ejecutar el notebook.")
else:
    FALTAN_MCD = ["(MCD sin respuesta)"]

In [8]:
def clima_sitio(s):
    medias, minimos, maximos, presion = [], [], [], []
    for ls in LS_MARTE:
        if not os.path.exists(ruta_mcd(s["id"], ls)):
            return None
        bl = leer_ascii(open(ruta_mcd(s["id"], ls)).read())
        t = np.array(bl[0]["y"]); t = t[(t > 50) & (t < 400)]
        if t.size < 10:
            return None
        medias.append(t.mean()); minimos.append(t.min()); maximos.append(t.max())
        if len(bl) > 1:
            presion.append(np.mean(bl[1]["y"]))
    medias, minimos, maximos = map(np.array, (medias, minimos, maximos))
    return {"t_media": medias.mean() - 273.15, "t_frio": medias.min() - 273.15, "t_calido": medias.max() - 273.15,
            "amplitud_anual": medias.max() - medias.min(), "osc_diaria": (maximos - minimos).mean(),
            "t_min": minimos.min() - 273.15, "t_max": maximos.max() - 273.15,
            "presion_pa": np.mean(presion) if presion else np.nan}

filas_m = []
if MCD_OK:
    for s in marte:
        c = clima_sitio(s)
        if c:
            filas_m.append({"id": s["id"], "nombre": s["nombre"], "tipo": s["tipo"], "lat": s["lat"], "lon": s["lon"],
                            **{k: round(float(v), 1) for k, v in c.items()}})
clima_m = pd.DataFrame(filas_m)
if len(clima_m):
    clima_m.to_csv(f"{RES}/clima_marte_sitios.csv", index=False)
    display(clima_m.sort_values("t_media")[["nombre", "tipo", "lat", "t_media", "t_min", "t_max", "amplitud_anual", "osc_diaria", "presion_pa"]])

## 6. Comparación por percentiles relativos (Marte ↔ Tierra)
Cada sitio de Marte se compara con las regiones del perfil Marte y los análogos conocidos de Marte. Se usa el percentil (0–100) de cada
medida dentro de su propio planeta. Después se combina con la distancia de topografía (`topografia_sitios_v1`).

In [9]:
t_m = tierra[(tierra.cuerpo == "marte")].dropna(subset=MEDIDAS).copy()
pares = pd.DataFrame()
if len(clima_m) and len(t_m):
    pm = clima_m.set_index("id")[MEDIDAS].rank(pct=True) * 100
    pt = t_m.set_index("region_id")[MEDIDAS].rank(pct=True) * 100
    filas = []
    for sid, ps in pm.iterrows():
        dif = (pt - ps).abs() / 100
        for rid, d in dif.iterrows():
            filas.append({"sitio_id": sid, "region_id": rid, **{f"dif_{m}": round(d[m], 3) for m in MEDIDAS},
                          "distancia_clima": round(d.mean(), 3)})
    pares = pd.DataFrame(filas)
    nom_s = clima_m.set_index("id")[["nombre", "tipo"]].rename(columns={"nombre": "sitio", "tipo": "tipo_sitio"})
    nom_t = t_m.set_index("region_id")[["region", "tipo"]].rename(columns={"tipo": "tipo_tierra"})
    pares = pares.join(nom_s, on="sitio_id").join(nom_t, on="region_id")
    ruta_topo = f"{BASE}/04_resultados/topografia_sitios_v1/similitud_sitio_tierra.csv"
    if os.path.exists(ruta_topo):
        topo = pd.read_csv(ruta_topo)[["sitio_id", "region_id", "distancia"]].rename(columns={"distancia": "distancia_topografia"})
        pares = pares.merge(topo, on=["sitio_id", "region_id"], how="left")
        pares["distancia_total"] = pares[["distancia_clima", "distancia_topografia"]].mean(axis=1, skipna=False).round(3)
        sin_topo = pares.distancia_topografia.isna().mean()
        if sin_topo > 0:
            print(f"⚠ {sin_topo:.0%} de las parejas no tienen distancia de topografía (quedan al final de la lista)")
    else:
        print("⚠ No encuentro topografia_sitios_v1: solo se usa el clima"); pares["distancia_topografia"] = np.nan
        pares["distancia_total"] = pares.distancia_clima
    pares = pares.sort_values(["sitio_id", "distancia_total"])
    pares.to_csv(f"{RES}/similitud_clima_marte.csv", index=False)

    mejor = lambda col: pares.dropna(subset=[col]).sort_values(col).groupby("sitio_id").head(1).set_index("sitio_id")
    res_s = clima_m.set_index("id")[["nombre", "tipo", "t_media"]].join(
        mejor("distancia_clima")[["region", "distancia_clima"]].rename(columns={"region": "mas_parecida_clima"})).join(
        mejor("distancia_total")[["region", "distancia_clima", "distancia_topografia", "distancia_total"]]
        .rename(columns={"region": "mas_parecida_total", "distancia_clima": "clima_de_esa", "distancia_topografia": "topografia_de_esa"}))
    res_s["hay_parecido"] = np.where(res_s.distancia_total <= UMBRAL_PARECIDO, "Sí", "No")
    res_s.to_csv(f"{RES}/clima_topografia_sitios_marte.csv")
    display(res_s.sort_values("distancia_total"))
    print("Sitios con parecido (clima + topografía ≤ 0,2):", f"{(res_s.hay_parecido == 'Sí').sum()}/{len(res_s)}")
    display(res_s.groupby("mas_parecida_total").size().sort_values(ascending=False).rename("veces_mas_parecida"))

### 6.1 Análogos conocidos de Marte: ¿a qué sitios se parecen por clima?

In [10]:
if len(pares):
    an = pares[pares.tipo_tierra == "analogo_conocido"]
    resumen_an = (an.sort_values("distancia_clima").groupby("region")
                  .agg(sitio_mas_parecido_clima=("sitio", "first"), distancia_clima=("distancia_clima", "first"),
                       sitios_parecidos_clima=("distancia_clima", lambda v: int((v <= UMBRAL_PARECIDO).sum())))
                  .sort_values("distancia_clima"))
    resumen_an = resumen_an.join(tierra.set_index("region")[["t_media", "t_min", "osc_diaria", "precip_mm"]])
    resumen_an.to_csv(f"{RES}/analogos_marte_clima.csv")
    display(resumen_an)

    # Valores absolutos: cuánto más frío es Marte que su pareja de la Tierra
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(t_m.t_media, t_m.osc_diaria, c=np.where(t_m.tipo == "analogo_conocido", "tab:red", "tab:green"), s=60)
    for r in t_m.itertuples():
        ax.annotate(str(r.region)[:20], (r.t_media, r.osc_diaria), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.scatter(clima_m.t_media, clima_m.osc_diaria, c="tab:orange", s=25, label="sitios de Marte")
    ax.scatter([], [], c="tab:green", label="Tierra: regiones candidatas"); ax.scatter([], [], c="tab:red", label="Tierra: análogos conocidos")
    ax.set_xlabel("temperatura media anual (°C)"); ax.set_ylabel("oscilación diaria media (°C)"); ax.grid(alpha=0.3); ax.legend()
    ax.set_title("Clima de los sitios de Marte frente a la Tierra (valores absolutos)")
    plt.show()

## 7. Capa para la web y registro

In [11]:
def limpio(v):
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    if isinstance(v, np.integer): return int(v)
    return v

TERMINADO = not FALTAN_M2 and not FALTAN_MCD
salida = {"version": VERSION,
          "unidades": {"temperaturas": "°C", "amplitud_anual y osc_diaria": "°C (K)", "precip_mm": "mm/año", "presion_pa": "Pa",
                       "distancias": "0–1, menor es más parecido (clima: diferencia media de percentiles)"},
          "tierra": {r["region_id"]: {k: limpio(v) for k, v in r.items() if k != "region_id"} for r in tierra.to_dict("records")},
          "marte_sitios": {}}
if len(pares):
    top = pares.groupby("sitio_id").head(5)
    for r in clima_m.to_dict("records"):
        salida["marte_sitios"][r["id"]] = {**{k: limpio(v) for k, v in r.items() if k != "id"},
            "top5_tierra": [{k: limpio(v) for k, v in p.items() if k in ("region_id", "region", "tipo_tierra", "distancia_clima",
                             "distancia_topografia", "distancia_total")} for p in top[top.sitio_id == r["id"]].to_dict("records")]}
json.dump(salida, open(f"{WEB}/clima.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, allow_nan=False)
if TERMINADO:
    entrada = json.load(open(f"{BASE}/06_web/manifest.json", encoding="utf-8"))
    entrada.setdefault("capas", {})["clima"] = {"version": VERSION, "carpeta": VERSION, "archivo": f"{VERSION}/clima.json"}
    json.dump(entrada, open(f"{BASE}/06_web/manifest.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
registro = {"version": VERSION, "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
            "datos": {"tierra": "NASA MERRA-2 2014–2023: clima de EDA_analogos_v1.3 (M2TMNXSLV, M2TMNXFLX) y M2SMNXSLV (T2MMAX, T2MMIN)",
                      "marte": "Mars Climate Database (LMD/ESA/CNES), interfaz web, escenario de polvo climatológico, topografía de alta resolución"},
            "parametros": {"anios": [min(ANIOS), max(ANIOS)], "gradiente_k_km": GRADIENTE_K_KM, "ls_marte": LS_MARTE,
                           "altura_marte_m": ALTURA_MARTE_M, "medidas_comparacion": MEDIDAS, "umbral_parecido": UMBRAL_PARECIDO},
            "progreso": {"merra2_meses_faltan": len(FALTAN_M2), "mcd_consultas_faltan": len(FALTAN_MCD)},
            "limitaciones": ["MERRA-2 a ~50 km: un análogo pequeño (un valle, un cráter) recibe el clima de toda la celda, corregido solo por altitud",
                             "la comparación es relativa (percentiles): Marte es mucho más frío que cualquier lugar de la Tierra",
                             "Marte solo en 4 estaciones (Ls 0, 90, 180, 270); la amplitud anual es aproximada",
                             "la MCD es un modelo, no medidas; incluye el escenario de polvo medio, no tormentas globales",
                             "la Luna no tiene clima: no se compara"]}
json.dump(registro, open(f"{RES}/registro_{VERSION}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("✅ Terminado y registrado en el manifest." if TERMINADO else "⏸ Guardado parcial: vuelve a ejecutar el notebook para completar.")

⏸ Guardado parcial: vuelve a ejecutar el notebook para completar.


In [1]:
!pip install -q zarr
!pip install -q -U huggingface_hub zarr xarray

In [2]:

import xarray as xr
from huggingface_hub import HfFileSystem
fs = HfFileSystem()
print(fs.ls("datasets/ananyo01/ARCO-MACDA", detail=False))
ds = xr.open_zarr(fs.get_mapper("datasets/ananyo01/ARCO-MACDA/macda_combined.zarr"), decode_times=False)
print(ds)
for v in ds.data_vars:
    print(v, ds[v].dims, ds[v].attrs.get("long_name"), ds[v].attrs.get("units"))
print({c: ds[c].values[:5] for c in ds.coords})

['datasets/ananyo01/ARCO-MACDA/macda_combined.zarr', 'datasets/ananyo01/ARCO-MACDA/.gitattributes', 'datasets/ananyo01/ARCO-MACDA/README.md']
<xarray.Dataset> Size: 216GB
Dimensions:  (time: 96480, lat: 36, lon: 72, lev: 35)
Coordinates:
  * time     (time) float64 772kB 0.08333 0.1667 0.25 ... 8.04e+03 8.04e+03
  * lat      (lat) float32 144B 87.5 82.5 77.5 72.5 ... -72.5 -77.5 -82.5 -87.5
  * lon      (lon) float32 288B -180.0 -175.0 -170.0 ... 165.0 170.0 175.0
  * lev      (lev) float32 140B 0.9995 0.9982 0.996 ... 2.892e-05 1.087e-05
Data variables: (12/14)
    co2ice   (time, lat, lon) float32 1GB dask.array<chunksize=(120, 36, 72), meta=np.ndarray>
    coldust  (time, lat, lon) float32 1GB dask.array<chunksize=(120, 36, 72), meta=np.ndarray>
    dustmmr  (time, lev, lat, lon) float32 35GB dask.array<chunksize=(120, 35, 36, 72), meta=np.ndarray>
    geop     (time, lev, lat, lon) float32 35GB dask.array<chunksize=(120, 35, 36, 72), meta=np.ndarray>
    Ls       (time) float32 386

## 8. Conclusiones
*Rellenar tras la ejecución.*